# Time series Classification
This notebook contains a comprehensive workflow for time series classification of ECG data to detect ischemic events. 

It includes:

- Data loading and preprocessing, including label generation and feature engineering.
- Multiple classification models: Logistic Regression, Support Vector Machine (SVM), Gradient Boosting, and Random Forest, with hyperparameter tuning and evaluation (accuracy, ROC, confusion matrix, etc.).
- Extraction of time series motifs and anomalies using matrix profiles and motif discovery.
- Feature extraction from ECG signals, including morphology and rhythm features.
- Classification using features derived from motifs and matrix profiles.
- Shapelet-based time series classification using tslearn, including visualization of learned shapelets and their locations in the ECG signals.
- Visualizations for model performance, feature importance, and motif/shapelet analysis.

The notebook is structured to compare traditional feature-based classifiers with advanced time series methods, providing both code and visual insights for each step.

In [ ]:
import sys
import os
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "../.."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
import pandas as pd
from src.utils.labels_binary_classification import generate_ischemic_labels
import numpy as np
from tslearn.shapelets import LearningShapelets
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from tslearn.shapelets import grabocka_params_to_shapelet_size_dict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier


import matplotlib.pyplot as plt

from tslearn.preprocessing import TimeSeriesScalerMinMax

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    RocCurveDisplay
)

import stumpy

PROFILE_PATH = "../../data/processed/time_series_features.pkl"
DIAG_PATH = "../../data/raw/heart_diagnoses_1.csv"
LABELS_OUT = "../../data/processed/subject_to_labels_ischemic.csv"

df = pd.read_pickle(PROFILE_PATH)
df["subject_id"] = df["subject_id"].astype(str)

print("Patient profile loaded:", df.shape)

labels_df = generate_ischemic_labels(
    diagnoses_path=DIAG_PATH,
    output_path=LABELS_OUT
)
labels_df["subject_id"] = labels_df["subject_id"].astype(str)  # Ensure matching types

print("Labels generated:")
print(labels_df["label_ischemic"].value_counts())

# Remove ICD-derived features to avoid data leakage
ICD_MACRO_COLS = [
    "ACS_acute_ischemic",
    "Chronic_ischemic",
    "Heart_failure",
    "Valvular_nonrheumatic",
    "Myocarditis_or_cardiomyopathy",
    "Pericardial_disease",
    "Endocarditis",
    "Cardiac_arrest",
    "Arrhythmias_or_conduction",
]

clf_df = df.drop(columns=[c for c in ICD_MACRO_COLS if c in df.columns])

# Merge labels with features
clf_df = clf_df.merge(labels_df[["subject_id", "label_ischemic"]], on="subject_id", how="inner")

print("Final classification dataset:", clf_df.shape)
print(f"Label distribution:\n{clf_df['label_ischemic'].value_counts()}")

# Convert 'gender' to boolean 
clf_df['gender'] = clf_df['gender'].map({'M': 1, 'F': 0})

clf_df

In [ ]:
def evaluate_binary(model, Xtr, ytr, Xte, yte):
    """
    Evaluate a binary classification model on training and test data.

    Parameters:
        model: Fitted classifier with predict and predict_proba methods (e.g., scikit-learn estimator).
        Xtr: Training features (array-like or DataFrame).
        ytr: Training labels (array-like or Series).
        Xte: Test features (array-like or DataFrame).
        yte: Test labels (array-like or Series).

    Returns:
        results: dict
            Dictionary with accuracy, balanced accuracy, precision, recall, F1, and ROC-AUC for both train and test sets.
        tuple:
            (ytr_pred, yte_pred, ytr_proba, yte_proba)
            - ytr_pred: Predicted class labels for training set.
            - yte_pred: Predicted class labels for test set.
            - ytr_proba: Predicted probabilities for positive class (train).
            - yte_proba: Predicted probabilities for positive class (test).
    """
    # Predict class labels (0/1) for train and test sets
    ytr_pred = model.predict(Xtr)
    yte_pred = model.predict(Xte)

    # Predict probabilities for ROC-AUC calculation
    ytr_proba = model.predict_proba(Xtr)[:, 1]
    yte_proba = model.predict_proba(Xte)[:, 1]

    # Collect evaluation metrics for both train and test sets
    results = {
        "train_accuracy": accuracy_score(ytr, ytr_pred),
        "test_accuracy": accuracy_score(yte, yte_pred),
        "train_balanced_accuracy": balanced_accuracy_score(ytr, ytr_pred),
        "test_balanced_accuracy": balanced_accuracy_score(yte, yte_pred),
        "train_precision": precision_score(ytr, ytr_pred),
        "test_precision": precision_score(yte, yte_pred),
        "train_recall": recall_score(ytr, ytr_pred),
        "test_recall": recall_score(yte, yte_pred),
        "train_f1": f1_score(ytr, ytr_pred),
        "test_f1": f1_score(yte, yte_pred),
        "train_roc_auc": roc_auc_score(ytr, ytr_proba),
        "test_roc_auc": roc_auc_score(yte, yte_proba),
    }
    # Return metrics and all predictions/probabilities for further analysis
    return results, (ytr_pred, yte_pred, ytr_proba, yte_proba)

## Features classification

## Logistic Regression

In [ ]:


TARGET = "label_ischemic"
DROP_COLS_METADATA = ['subject_id', 'hadm_id', 'smoothed_V5', 'long_title', 'icd_code', 'pNN50', 'SDNN']

# Fix: flatten the list so all columns are passed as a single list
X = clf_df.drop(columns=[TARGET] + DROP_COLS_METADATA)
y = clf_df[TARGET].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.30, # Splling the dataset in 70% training and 30% test
    stratify=y,
    random_state=42
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("Train:", X_train.shape, " Test:", X_test.shape)
print("Train label distribution:\n", y_train.value_counts(normalize=True))
print("Test  label distribution:\n", y_test.value_counts(normalize=True))

In [ ]:
from sklearn.model_selection import GridSearchCV

# Define the parameter grid for LogisticRegression
param_grid = {
    'model__C': [0.01, 0.1, 1.0, 10.0, 100.0],
    'model__penalty': ['l2'],
    'model__solver': ['lbfgs', 'liblinear'],
    'model__class_weight': ['balanced', None],
    'model__max_iter': [1000, 2000,3000,5000]
}

# Create the pipeline as before
lr_clf = Pipeline(steps=[
    ("model", LogisticRegression(random_state=42))
])

# Set up GridSearchCV
grid_search = GridSearchCV(
    lr_clf,
    param_grid,
    cv=5,
    scoring='balanced_accuracy',
    n_jobs=-1,
    verbose=2
)

# Fit the grid search
grid_search.fit(X_train, y_train)

print("Best parameters found:", grid_search.best_params_)
print("Best balanced accuracy:", grid_search.best_score_)

In [ ]:
# Build and fit a new pipeline using the best hyperparameters found by grid_search
best_lr_clf = Pipeline(steps=[
    ("model", LogisticRegression(
        random_state=42,
        C=grid_search.best_params_["model__C"],
        penalty=grid_search.best_params_["model__penalty"],
        solver=grid_search.best_params_["model__solver"],
        class_weight=grid_search.best_params_["model__class_weight"],
        max_iter=grid_search.best_params_["model__max_iter"]
    ))
])
best_lr_clf.fit(X_train, y_train)

# Evaluate the optimized logistic regression model on train and test sets
lr_results, (y_train_pred, y_test_pred, y_train_proba, y_test_proba) = evaluate_binary(
    best_lr_clf, X_train, y_train, X_test, y_test
)

# Display the evaluation metrics in sorted order
pd.Series(lr_results).sort_index()

In [ ]:
# Extract and visualize the most important features from the trained logistic regression model

# Get the coefficients from the fitted logistic regression model
coef = best_lr_clf.named_steps["model"].coef_[0]

# Create a sorted Series of coefficients by absolute value (most influential features first)
coef_df = (
    pd.Series(coef, index=X.columns)
    .sort_values(key=np.abs, ascending=False)
)

# Display the top 10 most important coefficients
coef_df.head(10)

# Plot the top 10 coefficients as a horizontal bar chart
plt.figure(figsize=(8, 5))
coef_df.head(10).plot(kind="barh")
plt.title("Top-10 Logistic Regression Coefficients")
plt.gca().invert_yaxis()  # Highest coefficient at the top
plt.show()

The Logistic Regression model relies heavily on patient metadata (Gender, Age) to make decisions, with demographic factors outweighing physiological signal features. While the model successfully identified T-wave depression/inversion (via the negative coefficient for T-wave amplitude) as a risk factor, the counter-intuitive weighting of Age suggests potential biases in the training data distribution.

In [ ]:
from sklearn.metrics import precision_recall_curve

# Compute precision, recall, and thresholds for different decision boundaries
prec, rec, thr = precision_recall_curve(y_test, y_test_proba)

# Plot how precision and recall change as the decision threshold varies
plt.plot(thr, prec[:-1], label="Precision")
plt.plot(thr, rec[:-1], label="Recall")
plt.title("Precision and Recall vs Decision Threshold (Logistic Regression)")
plt.xlabel("Decision threshold")
plt.legend()
plt.show()

The Precision-Recall versus Decision Threshold plot for the Logistic Regression classifier illustrates the model's limited predictive capacity and poor calibration. The precision curve remains low throughout, starting around 0.49 and barely exceeding 0.55 at its peak, indicating that the model struggles to identify a reliable "safe zone" of confident predictions. At the default decision threshold of approximately 0.50, the precision and recall curves intersect, with both metrics stabilizing around 0.53–0.54, which aligns with the previously reported low F1-scores.

Most critically, the plot reveals significant instability at higher thresholds. Typically, precision should improve as the threshold increases and the model becomes more selective. However, in this case, the precision curve becomes jagged and drops sharply as the threshold approaches 0.60. This anomaly suggests that the model's highest-confidence predictions are frequently incorrect (False Positives), highlighting a fundamental failure of the linear decision boundary to isolate distinct ischemic cases. Furthermore, high recall (>0.90) is only achievable at very low thresholds (<0.40), forcing a trade-off where nearly half of all positive flags are false alarms. This confirms that there is no optimal operating point for this model.

In [ ]:
from sklearn.model_selection import cross_val_score

# Perform 5-fold cross-validation using the best logistic regression model
# This provides a more robust estimate of model performance on unseen data
cv_scores_lr = cross_val_score(
    best_lr_clf,
    X_train,
    y_train,
    cv=5,
    scoring="balanced_accuracy"
)

# Print the mean and standard deviation of the balanced accuracy across folds
print("Logistic Regression - 5-fold CV balanced accuracy")
print(f"Mean = {cv_scores_lr.mean():.3f}")
print(f"Std  = {cv_scores_lr.std():.3f}")

In [ ]:
# Confusion matrix (test) for best logistic regression model
cm = confusion_matrix(y_test, y_test_pred, labels=[0, 1])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[0, 1])
disp.plot()
plt.title("Best Logistic Regression - Confusion Matrix (test)")
plt.show()

print("\nClassification report (test):")
print(classification_report(y_test, y_test_pred, target_names=["Non-ischemic (0)", "Ischemic (1)"]))

# ROC curve (test) for best logistic regression model
RocCurveDisplay.from_estimator(best_lr_clf, X_test, y_test)
plt.title("Best Logistic Regression - ROC curve (test)")
plt.show()

The model achieved a precision of 0.56 and a recall of 0.55 for the non-ischaemic class. This indicates that while the model correctly identified 55\% of healthy subjects (specificity), it struggled to confidently differentiate them from ischaemic cases. Approximately 44\% of the subjects predicted as "healthy" were, in fact, ischaemic (false negatives), which limits the model's reliability in ruling out disease. For the critical ischaemic class, the model demonstrated a precision of 0.53 and a recall (sensitivity) of 0.54. A sensitivity of 0.54 implies that the classifier missed nearly half (46\%) of the actual ischaemia cases. Additionally, the low precision of 0.53 suggests that when the model flagged a patient as ischaemic, the likelihood of that diagnosis being correct was almost equivalent to chance.

## Support Vector Machine

In [ ]:
from sklearn.svm import SVC

# Create the SVM classifier
svm_clf = SVC(
    kernel="rbf",          # Radial Basis Function kernel
    C=1.0,                 # Regularization parameter
    gamma="scale",        # Kernel coefficient
    probability=True,     # Enable probability estimates
    class_weight="balanced",
    random_state=42
)

svm_clf.fit(X_train, y_train)

svm_results, (y_train_pred_svm, y_test_pred_svm, y_train_proba_svm, y_test_proba_svm) = evaluate_binary(
    svm_clf, X_train, y_train, X_test, y_test
)

pd.Series(svm_results).sort_index()

In [ ]:
prec, rec, thr = precision_recall_curve(y_test, y_test_proba)

plt.plot(thr, prec[:-1], label="Precision")
plt.plot(thr, rec[:-1], label="Recall")
plt.title("Precision and Recall vs Decision Threshold (SVM)")
plt.xlabel("Decision threshold")
plt.legend()
plt.show()

The SVM classifier exhibits performance limitations nearly identical to the Logistic Regression model, confirming a failure to establish a robust separation boundary. The precision curve remains stagnant around $0.50$ for the majority of the decision range, never reaching a zone of high confidence. Crucially, increasing the decision threshold beyond $0.50$ does not improve precision as expected; instead, the metric becomes unstable and degrades, indicating that even the model's "strictest" predictions are unreliable. The intersection with recall occurs near the default threshold ($0.50$), where both metrics plateau at approximately $0.53$, offering only marginal improvement over a random baseline.

In [ ]:
# Plot the confusion matrix for SVM (test)
cm_svm = confusion_matrix(y_test, y_test_pred_svm, labels=[0, 1])
disp_svm = ConfusionMatrixDisplay(confusion_matrix=cm_svm, display_labels=[0, 1])
disp_svm.plot()
plt.title("SVM - Confusion Matrix (test)")
plt.show()

print("\nSVM Classification report (test):")
print(classification_report(y_test, y_test_pred_svm, target_names=["Non-ischemic (0)", "Ischemic (1)"]))

# PLot the ROC curve for SVM (test)
RocCurveDisplay.from_predictions(y_test, y_test_proba_svm)
plt.title("SVM - ROC curve (test)")
plt.show()

The model exhibits a slight bias towards the positive class, achieving a Recall of $0.56$. This means the classifier correctly identified $56\%$ of actual Ischemic cases. However, the Precision is low at $0.51$, indicating that nearly half of the patients predicted as Ischemic were actually false positives. The model also struggled with the negative class, obtaining a Recall of $0.50$. This suggests that the classifier misdiagnosed $50\%$ of healthy (Non-ischemic) individuals as having the condition. The F1-scores, which are $0.52$ for Non-ischemic cases and $0.54$ for Ischemic cases, confirm that the model lacks robustness. While the SVM shows a marginal preference for detecting Ischemic cases over healthy ones, the overall accuracy of $0.53$ implies that the current decision boundary is ineffective, performing only slightly better than random chance.

## Gradient boosting Classifier

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import GradientBoostingClassifier

# Define the hyperparameter grid for tuning the GradientBoostingClassifier.
# Each key corresponds to a parameter, and the list contains the values to try during grid search.
param_grid_gb = {
    'n_estimators': [100, 200, 300],      # Number of boosting stages (trees)
    'learning_rate': [0.01, 0.05, 0.1],  # Step size shrinkage used in update to prevent overfitting
    'max_depth': [3, 5, 6],              # Maximum depth of each tree
    'subsample': [0.8, 1.0],             # Fraction of samples used for fitting each base learner
    'min_samples_split': [2, 5, 10],     # Minimum number of samples required to split an internal node
    'min_samples_leaf': [1, 3, 5]        # Minimum number of samples required to be at a leaf node
}

# Initialize the base GradientBoostingClassifier with a fixed random seed for reproducibility.
gb_clf_base = GradientBoostingClassifier(random_state=42)

# Set up GridSearchCV to perform exhaustive search over the specified parameter grid.
# - cv=5: 5-fold cross-validation
# - scoring='balanced_accuracy': Use balanced accuracy as the evaluation metric
# - n_jobs=-1: Use all available CPU cores for parallel processing
# - verbose=2: Print progress messages during fitting
grid_search_gb = GridSearchCV(
    gb_clf_base,
    param_grid_gb,
    cv=5,
    scoring='balanced_accuracy',
    n_jobs=-1,
    verbose=2
)

# Fit the grid search to the training data to find the best hyperparameter combination.
grid_search_gb.fit(X_train, y_train)

# Output the best hyperparameters and the corresponding balanced accuracy score.
print("Best parameters found:", grid_search_gb.best_params_)
print("Best balanced accuracy:", grid_search_gb.best_score_)

# Retrieve the best estimator from the grid search and refit it on the full training set.
gb_best = grid_search_gb.best_estimator_
gb_best.fit(X_train, y_train)

# Evaluate the optimized Gradient Boosting model using the custom evaluation function.
gb_results, (gb_y_train_pred, gb_y_test_pred, gb_y_train_proba, gb_y_test_proba) = evaluate_binary(
    gb_best, X_train, y_train, X_test, y_test
)

# Display the evaluation metrics in sorted order for easier interpretation.
print(pd.Series(gb_results).sort_index())

In [ ]:
# Perform 5-fold cross-validation using the best Gradient Boosting model.
# This provides a more robust estimate of model performance by evaluating it on different splits of the training data.
cv_scores_gb = cross_val_score(
    gb_best,           # The optimized GradientBoostingClassifier
    X_train,           # Training features
    y_train,           # Training labels
    cv=5,              # Number of cross-validation folds
    scoring="balanced_accuracy"  # Use balanced accuracy to account for class imbalance
)

# Print the mean and standard deviation of the balanced accuracy across the 5 folds.
print("GB - 5-fold CV balanced accuracy")
print(f"Mean = {cv_scores_gb.mean():.3f} | Std = {cv_scores_gb.std():.3f}")

In [ ]:
# Confusion matrix (test)
cm_rf = confusion_matrix(y_test, gb_y_test_pred, labels=[0, 1])
disp_rf = ConfusionMatrixDisplay(confusion_matrix=cm_rf, display_labels=[0, 1])
disp_rf.plot()
plt.title("GradientBoostingClassifier - Confusion Matrix (test)")
plt.show() 
print("\nGradient Boosting Classification report (test):")
print(classification_report(y_test, gb_y_test_pred, target_names=["Non-ischemic (0)", "Ischemic (1)"]))
# ROC curve (test)
RocCurveDisplay.from_predictions(y_test, gb_y_test_proba)
plt.title("GradientBoostingClassifier - ROC curve (test)")
plt.show()

Based on the results, we can conclude that the model performed best on the negative class, achieving a Recall of $0.64$ and a Precision of $0.58$. This indicates that the model is relatively effective at accurate ruling out the disease in healthy patients, reflected in an F1-score of $0.61$. However, detecting the positive class remains challenging. While the Precision for ischemia is moderate at $0.57$—meaning that predictions for ischemia are correct $57\%$ of the time—the Recall is lower at $0.51$. This suggests that the model misses approximately $49\%$ of actual ischemic cases, resulting in a high rate of false negatives.In contrast to the results from the Support Vector Machine (SVM) and Logistic Regression, the Gradient Boosting model demonstrates a slightly superior predictive power with an overall accuracy of $0.58$. However, unlike the previous iteration, the performance is not perfectly balanced; there is a noticeable bias toward the non-ischemic class (F1-score $0.61$ vs $0.54$). Although the macro-averaged metrics ($0.58$) indicate moderate general capability, further optimization is clearly needed to enhance the sensitivity for clinical applications.

## Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Initialize a baseline Random Forest classifier with specific hyperparameters:
rf_clf = RandomForestClassifier(
    n_estimators=30,        # A moderate number of trees to reduce variance compared to a single decision tree
    criterion="gini",       # Use the Gini impurity to measure the quality of splits
    max_features=3,         # Randomly select 3 features at each split to introduce diversity among trees
    max_depth=4,            # Limit the maximum depth of each tree to prevent overfitting
    min_samples_split=2,    # Minimum number of samples required to split an internal node (default)
    min_samples_leaf=8,     # Minimum number of samples required at a leaf node for more stable and generalizable predictions
    bootstrap=True,         # Use bootstrap samples (sampling with replacement) to build each tree
    class_weight="balanced",# Adjust weights inversely proportional to class frequencies to handle class imbalance
    random_state=42         # Set random seed for reproducibility
)

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

# Define the hyperparameter search space for RandomizedSearchCV.
# Each key is a parameter name, and the values are lists or distributions to sample from.
param_dist = {
    "n_estimators": [30, 50, 100],           # Number of trees in the forest
    "max_depth": [2, 4, 6, 8, None],         # Maximum depth of each tree (None means unlimited)
    "max_features": [2, 3, "sqrt"],          # Number of features to consider at each split
    "min_samples_split": randint(2, 20),     # Minimum samples required to split a node (randomly sampled)
    "min_samples_leaf": randint(5, 20),      # Minimum samples required at a leaf node (randomly sampled)
    "bootstrap": [True],                     # Whether bootstrap samples are used when building trees
    "criterion": ["gini", "entropy"],        # Function to measure the quality of a split
    "class_weight": ["balanced", None]       # Adjust weights for class imbalance or use default
}

In [ ]:
rf_base = rf_clf  # Use the previously defined baseline Random Forest as the starting estimator

# Set up RandomizedSearchCV to search for the best hyperparameters for the Random Forest.
# - estimator: the base Random Forest model
# - param_distributions: the hyperparameter search space defined above
# - n_iter=30: sample 30 different combinations of hyperparameters
# - scoring='balanced_accuracy': use balanced accuracy to account for class imbalance
# - cv=5: use 5-fold cross-validation for evaluation
# - n_jobs=-1: use all available CPU cores for parallel processing
# - random_state=42: ensure reproducibility of the search
rf_search = RandomizedSearchCV(
    estimator=rf_base,
    param_distributions=param_dist,
    n_iter=30,
    scoring="balanced_accuracy",
    cv=5,
    n_jobs=-1,
    random_state=42
)

# Fit the randomized search to the training data to find the best hyperparameter combination.
rf_search.fit(X_train, y_train)

# Print the best hyperparameters found during the search.
print("Best parameters found:")
print(rf_search.best_params_)

In [ ]:
# Retrieve the best Random Forest model found by RandomizedSearchCV.
rf_best = rf_search.best_estimator_

# Fit the best Random Forest model to the full training data.
rf_best.fit(X_train, y_train)

# Evaluate the tuned Random Forest model on both training and test sets using the custom evaluation function.
rf_best_results, (rfb_y_train_pred, rfb_y_test_pred, rfb_y_train_proba, rfb_y_test_proba) = evaluate_binary(
    rf_best, X_train, y_train, X_test, y_test
)

# Display the evaluation metrics in sorted order for easier interpretation.
pd.Series(rf_best_results).sort_index()

In [ ]:
# comparison with baseline RF and RF with tuned hyperparameters (RandomizedSearchCV)
rf_baseline = rf_clf  # Base Random Forest model defined earlier
rf_baseline.fit(X_train, y_train)

# Evaluate the baseline Random Forest model on training and test sets
rf_base_results, _ = evaluate_binary(
    rf_baseline, X_train, y_train, X_test, y_test
)

# Compare baseline and tuned Random Forest results
comparison_rf = pd.DataFrame({
    "Baseline RF": rf_base_results,
    "Tuned RF": rf_best_results
})

comparison_rf.sort_index()


In [ ]:
# Feature Importance for Random Forest
feat_importances = pd.Series(rf_best.feature_importances_, index=X.columns)
feat_importances.nlargest(10).plot(kind='barh', title='Feature Importance - Random Forest')
plt.gca().invert_yaxis()
plt.show()

The Random Forest effectively combines patient demographics (Age) with morphological changes (R-peaks, T-waves) and non-linear rhythm dynamics (RMSSD). This multi-faceted approach explains why it generally outperforms simpler linear models that fail to integrate these distinct feature types.

In [ ]:
# Confusion matrix (test)
cm_rf = confusion_matrix(y_test, rfb_y_test_pred, labels=[0, 1])
disp_rf = ConfusionMatrixDisplay(confusion_matrix=cm_rf, display_labels=[0, 1])
disp_rf.plot()
plt.title("Tuned Random Forest - Confusion Matrix (test)")
plt.show() 
print("\nTuned Random Forest Classification report (test):")
print(classification_report(y_test, rfb_y_test_pred, target_names=["Non-ischemic (0)", "Ischemic (1)"]))
# ROC curve (test)
RocCurveDisplay.from_predictions(y_test, rfb_y_test_proba)
plt.title("Tuned Random Forest - ROC curve (test)")
plt.show()


The Tuned Random Forest classifier was evaluated on a test set consisting of 383 instances, with a balanced distribution between Non-ischaemic ($n=197$) and Ischaemic ($n=186$) classes. The model achieved an overall accuracy of 0.54, which indicates a limited ability to differentiate between the two classes. Despite hyperparameter tuning, the performance metrics suggest that the model performs only marginally better than a random baseline, presenting significant challenges for deployment in a diagnostic capacity. A critical deficiency is noted in the sensitivity of the Ischaemic class, which stands at 0.56. This translates to a False Negative Rate of 44\%, meaning nearly half of the positive cases in the test set were misclassified as Non-ischaemic. In a clinical context, this lack of sensitivity poses a substantial risk, as it can lead to missed diagnoses. The precision for the Ischaemic class is 0.53, indicating a 47\% False Positive Rate among the predicted positive cases. While this is less critical than missed diagnoses, the low precision undermines the reliability of positive flags, potentially resulting in unnecessary follow-up procedures for healthy individuals. The F1-score for the positive class is 0.54, reflecting a poor balance between precision and recall. This metric confirms that the model struggles to capture the complexity of the feature space necessary for effectively isolating Ischaemic instances.

### ROC curves comparison

In [ ]:

fig, ax = plt.subplots(figsize=(7, 6))

RocCurveDisplay.from_predictions(
    y_test, y_test_proba,
    name="Logistic Regression",
    ax=ax
)

RocCurveDisplay.from_predictions(
    y_test, y_test_proba_svm,
    name="SVM",
    ax=ax
)

RocCurveDisplay.from_predictions(
    y_test, gb_y_test_proba,
    name="Gradient Boosting",
    ax=ax
)

RocCurveDisplay.from_predictions(
    y_test, rfb_y_test_proba,
    name="Random Forest",
    ax=ax
)

ax.plot([0, 1], [0, 1], "k--", label="Random guess")
ax.set_title("ROC Curves Comparison (Test set)")
ax.legend()
plt.show()


The image presents a Receiver Operating Characteristic (ROC) curve comparison for four machine learning models evaluated on a test set. The x-axis shows the False Positive Rate (1 − Specificity), and the y-axis shows the True Positive Rate (Sensitivity). A dashed diagonal black line represents a random classifier (AUC = 0.50), which serves as the baseline for no predictive skill.
Model Performance (Legend) The legend lists the models in descending order of their Area Under the Curve (AUC) scores, which measure their overall ability to distinguish between classes:
- **Gradient Boosting** (green line), AUC = 0.63: This model achieves the best performance among the four. Its curve bends most prominently toward the top-left corner, indicating a more favorable balance between sensitivity and specificity than the other models.
- **SVM** (orange line), AUC = 0.56: The Support Vector Machine performs slightly better than random chance but remains relatively close to the diagonal.
- **Random Forest** (red line), AUC = 0.55: This model’s performance is comparable to that of the SVM and only marginally better than the random baseline, indicating limited predictive capability.
- **Logistic Regression** (blue line), AUC = 0.53: This is the lowest-performing model. Its curve lies close to the diagonal, suggesting it offers minimal improvement over random guessing in discriminating between classes.

## Time series motifs

In [ ]:
from scipy.stats import skew
from scipy.stats import kurtosis
# Get the profile matrices of each patient
fs = 500
features_list = []
profile_matrices = []
patient_beats = {}
patient_anomalies = {}

m = int(0.8 * fs)

# Ensure m is defined (e.g., m = 200 for typical heartbeat length at 100Hz)
# m = 200 

for subject_id, data in df.groupby('subject_id'):
    # Ensure signal is float for precision
    signal = np.array(data['smoothed_V5'].iloc[0], dtype=np.float64)

    if len(signal) < m:
        print(f"Signal for subject {subject_id} is shorter than m={m}. Skipping.")
        continue

    # 1. Compute Matrix Profile
    profile = stumpy.stump(signal, m)

    # 2. Get Top Motifs (The "Representative Beats")
    # We increase max_motifs to capture the repeating rhythm
    motif_distances, motif_indices = stumpy.motifs(signal, profile[:, 0], max_motifs=5, min_neighbors=3)
    
    # Store the location of the best motif (most common beat shape)
    if len(motif_indices) > 0:
        best_motif_idx = motif_indices[0][0]
        patient_beats[subject_id] = motif_indices[0]
        
        # Extract the actual waveform of the "Standard Beat" for this patient
        representative_beat = signal[best_motif_idx : best_motif_idx + m]
    else:
        # Fallback if no motif found (rare)
        best_motif_idx = 0
        representative_beat = signal[0:m]


    # 3. Anomaly detection (Discords)
    discord_indices = np.argpartition(profile[:, 0], -3)[-3:]
    discord_indices = discord_indices[np.argsort(profile[discord_indices, 0])[::-1]]
    discord_scores = profile[discord_indices, 0]
    patient_anomalies[subject_id] = list(zip(discord_indices, discord_scores))

    # 4. Features Extraction
    P = profile[:, 0].astype(np.float64)
    gap_padding = np.full(m - 1, np.nan)

    P_padded = np.concatenate((P, gap_padding))
    
    #Save gapped profile if needed
    profile_matrices.append(P_padded)

    I = profile[:, 1]
    
    # --- Helper: Ischemia usually depresses/elevates the ST segment ---
    # We estimate the "ST area" by looking at the last 30% of the representative beat
    # (assuming the beat is centered or starts with P-wave)
    st_segment_window = representative_beat[int(0.7 * m):] 

    features = {
        'subject_id': subject_id,
        
        # ==========================================
        # 1. MATRIX PROFILE FEATURES (Shape/Anomaly)
        # ==========================================
        'mp_min': np.min(P),       # How perfect is the best match?
        'mp_max': np.max(P),       # How distinct is the worst anomaly?
        'mp_mean': np.mean(P),     # General signal noise/irregularity
        'mp_std': np.std(P),       # Variability in beat shapes
        'mp_skew': skew(P),        # Distribution of matching difficulty
        'num_anomalies': np.sum(P > (np.mean(P) + 2*np.std(P))),

        # ==========================================
        # 2. ISCHEMIA SPECIFIC FEATURES (Morphology)
        # ==========================================
        # A. Global Signal Statistics
        # Ischemia (ST depression) often creates a "tail" in the voltage distribution
        'signal_skew': skew(signal),     
        'signal_kurtosis': kurtosis(signal),
        
        # B. Representative Beat Analysis (The "Motif")
        # Ischemia alters the voltage energy of the beat
        'beat_energy': np.sum(representative_beat**2),
        'beat_complexity': np.std(representative_beat), # Flatter beats (T-wave inversion) have lower std
        'beat_range': np.ptp(representative_beat),      # Peak-to-Peak amplitude

        # C. ST-Segment Proxies (Crucial for Ischemia)
        # Compare the ST-segment mean to the beat's global mean
        'st_deviation': np.mean(st_segment_window) - np.mean(representative_beat),
        'st_energy': np.sum(st_segment_window**2),
        
        # ==========================================
        # 3. RHYTHM & HRV FEATURES
        # ==========================================
        # A. Profile-based rhythm
        'avg_neighbor_dist': np.mean(np.abs(I - np.arange(len(I)))),
        
        # B. Motif-based HRV (Heart Rate Variability)
        # Calculate time distance between occurrences of the SAME motif
        # This approximates the RR-interval stability.
        
    }

    # Add HRV feature if we found enough motif neighbors
    if len(motif_indices) > 0 and len(motif_indices[0]) > 1:
        # Sort indices to find sequential occurrences
        sorted_beats = np.sort(motif_indices[0])
        # Differences between beats (RR intervals)
        rr_intervals = np.diff(sorted_beats)
        
        features['est_hr_mean'] = np.mean(rr_intervals) # Proxy for Heart Rate
        features['est_hrv_std'] = np.std(rr_intervals)  # Proxy for HRV (SDNN)
    else:
        features['est_hr_mean'] = 0
        features['est_hrv_std'] = 0

    features_list.append(features)




profile_matrices = np.array(profile_matrices)
profile_matrices.shape
    

In [ ]:
features_df = pd.DataFrame(features_list)
# Add the ischemic labels
features_df = features_df.merge(
    labels_df[["subject_id", "label_ischemic"]],
    on="subject_id",
    how="inner"
)

features_df

In [ ]:
# plot the Matrix Profile directly under your ECG signal for the first patient
subject_id = df['subject_id'].iloc[0]
signal = np.array(df[df['subject_id'] == subject_id]['smoothed_V5'].iloc[0], dtype=np.float64)
profile = profile_matrices[0, :len(signal) - m + 1]
plt.figure(figsize=(12, 6))
plt.subplot(2, 1, 1)
plt.plot(signal)
plt.title(f"ECG Signal for Subject {subject_id}")
plt.subplot(2, 1, 2)
plt.plot(profile, color='orange')
plt.title(f"Matrix Profile for Subject {subject_id}")
plt.xlabel("Time")
plt.tight_layout()
plt.show()

In [ ]:
# Visualize the top 3 anomalies for one specific patient
subject_to_plot = list(patient_beats.keys())[0] # Just pick the first patient
discords = patient_anomalies[subject_to_plot]  # List of (idx, score) tuples

# Retrieve that patient's signal again for plotting
signal = df[df['subject_id'] == subject_to_plot]['smoothed_V5'].iloc[0]

plt.figure(figsize=(12, 5))
plt.plot(signal, color='lightgray', label="Full ECG")

# Highlight the top 3 anomalies in different colors
colors = ['red', 'orange', 'purple']
for i, (idx, score) in enumerate(discords):
    plt.plot(range(idx, idx+m), signal[idx : idx+m], color=colors[i % len(colors)], linewidth=2, label=f"Anomaly {i+1} (Score {score:.2f})")

plt.title(f"Subject {subject_to_plot}: Top 3 Anomalies")
plt.legend()
plt.show()

The clustering of all three top anomalies at the very end of the signal suggests that the model identified a single, major irregular event rather than scattered issues. The morphology of this event—a sudden, extreme vertical spike that breaks the pattern of the previous heartbeats—differs significantly from the earlier periodic rhythm.

In [ ]:
indices = patient_beats[subject_to_plot]
plt.figure(figsize=(10, 5))
plt.title(f"Overlay of Dominant Motif for Subject {subject_to_plot}")

# Loop through every index found and plot that segment
for idx in indices:
    # Ensure we don't go out of bounds
    if idx + m < len(signal):
        plt.plot(signal[idx : idx+m], color='red', alpha=0.3) # alpha makes them transparent

plt.xlabel("Time (Samples)")
plt.ylabel("Amplitude")
plt.show()

In [ ]:
# Run a random forest classifier on the extracted features
TARGET = "label_ischemic"
X_feat = features_df.drop(columns=[TARGET, 'subject_id'])
y_feat = features_df[TARGET].astype(int)

# Split into train and test sets
Xf_train, Xf_test, yf_train, yf_test = train_test_split(
    X_feat, y_feat,
    test_size=0.30,
    stratify=y_feat,
    random_state=42
)

# Normalize features
scaler_feat = StandardScaler()
Xf_train = scaler_feat.fit_transform(Xf_train)
Xf_test = scaler_feat.transform(Xf_test)

param_dist = {
    "n_estimators": [30, 50, 100],
    "max_depth": [2, 4, 6, 8, None],
    "max_features": [2, 3, "sqrt"],
    "min_samples_split": randint(2, 20),
    "min_samples_leaf": randint(5, 20),
    "bootstrap": [True],
    "criterion": ["gini", "entropy"],
    "class_weight": ["balanced", None]
}

In [ ]:
rf_base = RandomForestClassifier(random_state=42)

rf_search = RandomizedSearchCV(
    estimator=rf_base,
    param_distributions=param_dist,
    n_iter=30,
    scoring="balanced_accuracy",
    cv=5,
    n_jobs=-1,
    random_state=42
)

rf_search.fit(X_train, y_train)

print("Best parameters found:")
print(rf_search.best_params_)

In [ ]:
rf_best = rf_search.best_estimator_

rf_best.fit(Xf_train, yf_train)

rf_best_results, (rfb_y_train_pred, rfb_y_test_pred,
                  rfb_y_train_proba, rfb_y_test_proba) = evaluate_binary(
    rf_best, Xf_train, yf_train, Xf_test, yf_test
)

pd.Series(rf_best_results).sort_index()

In [ ]:
rf_baseline = rf_base
rf_baseline.fit(Xf_train, yf_train)

rf_base_results, _ = evaluate_binary(
    rf_baseline, Xf_train, yf_train, Xf_test, yf_test
)

comparison_rf = pd.DataFrame({
    "Baseline RF (Features)": rf_base_results,
    "Tuned RF (Features)": rf_best_results
})

comparison_rf.sort_index()

In [ ]:
feat_importances = pd.Series(rf_best.feature_importances_, index=X_feat.columns)
feat_importances.nlargest(10).plot(kind='barh', title='Feature Importance - Random Forest')
plt.gca().invert_yaxis()
plt.show()

The Random Forest model prioritizes signal complexity over traditional clinical markers. Matrix Profile statistics dominate the decision process, with mp_std (variability), mp_max, and mp_mean occupying the top three spots. This indicates that the model relies primarily on the structural dissimilarity between heartbeats to detect ischemia. Distributional asymmetry also plays a key role, as evidenced by the high ranking of mp_skew and signal_skew. Notably, classic physiological indicators like st_energy and st_deviation are less influential, appearing lower in the hierarchy than these global complexity metrics.

In [ ]:
# Confusion matrix (test)
cm_rf = confusion_matrix(yf_test, rfb_y_test_pred, labels=[0, 1])
disp = ConfusionMatrixDisplay(confusion_matrix=cm_rf, display_labels=[0, 1])
disp.plot()
plt.title("Random Forest - Confusion Matrix (test)")
plt.show()

print("\nClassification report (test):")
print(classification_report(
    yf_test, rfb_y_test_pred,
    target_names=["Non-ischemic (0)", "Ischemic (1)"]
))

# ROC curve (test)
RocCurveDisplay.from_predictions(yf_test, rfb_y_test_proba)
plt.title("Random Forest - ROC curve (test)")
plt.show()

The model achieved an overall accuracy of $54\%$. In a binary classification task with a nearly balanced dataset ($51.4\%$ Class 0 vs. $48.6\%$ Class 1), a random guess would be expected to achieve an accuracy of approximately $50\%$. A result of $0.54$ suggests that the model has marginally surpassed the random baseline, showing a slight improvement in predictive capability compared to previous iterations. For the Ischaemic class (1), the model correctly identifies $68\%$ of the positive cases (Recall). This is a notable shift in behaviour, reducing the False Negative rate to $32\%$. However, this sensitivity comes at a cost: the Precision remains low at $0.52$, meaning that when the model flags a case as Ischaemic, it is incorrect $48\%$ of the time. The F1-Score of $0.59$ reflects this aggressive prediction strategy—the model captures more true cases but generates a high volume of false alarms. Performance for the negative class (Non-ischaemic) has suffered inversely. While the Precision is slightly higher at $0.57$, the Recall has dropped significantly to $0.40$. This indicates that the model now struggles to correctly identify healthy patients, misclassifying $60\%$ of true healthy signals as pathological. Effectively, the classifier has biased itself toward predicting "Ischaemia," improving safety (sensitivity) at the expense of specificity.

## Time series shaplets Classification

In [ ]:
# Get dataset dimensions
n_ts, ts_sz = clf_df.shape[0], clf_df['smoothed_V5'].iloc[0].shape[0]
n_classes = clf_df['label_ischemic'].nunique()

In [ ]:
from sklearn.model_selection import train_test_split
from tslearn.preprocessing import TimeSeriesScalerMeanVariance

# Prepare data for time-series classification
X = np.array(clf_df['smoothed_V5'].tolist())
y = np.array(clf_df['label_ischemic'].tolist())

# Reshape X to have a third dimension (n_samples, n_timesteps, n_features)
X = X.reshape(X.shape[0], X.shape[1], 1)


print(f"Original Shape: {X.shape}")
X = X[:, :1500, :]
print(f"New Shape (Time Sliced): {X.shape}") 

# Split into train and test sets with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
# Scale the data
scaler = TimeSeriesScalerMeanVariance()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
# Get the sampling rate from the data
sampling_rate_hz = 500 
shapelet_sizes = {
    20: 50,    # QRS Complex (40ms: sharp, distinctive)
    50: 30,    # P-Wave (100ms: atrial depolarization)
    100: 20,   # T-Wave (200ms: ventricular repolarization)
    200: 10,   # QT Interval (400ms: complete ventricular cycle)
}

print("Optimized ECG Sizes:", shapelet_sizes)

To classify the time series we use the LearningShapelets classifier, using the "Adam" optimizer.

The Adam optimizer is particularly effective for LearningShapelets because it addresses the main difficulties in optimizing shapelet parameters:
- Adaptive learning rates: Shapelets (subsequences) and classification weights can differ greatly in scale. Adam assigns a separate learning rate to each parameter, which helps prevent vanishing gradients that commonly affect shapelet learning.
- Faster convergence: By combining the momentum of SGD with the adaptive scaling of RMSProp, Adam moves through the complex, non-convex loss landscape more efficiently, often requiring fewer epochs to discover discriminative shapelets.
- Stability: Adam is less sensitive to the choice of initial learning rate and better handles the noisy gradients produced by sliding-window distance calculations than standard SGD.

In [ ]:
# Create the LearningShapelets classifier
shp_clf = LearningShapelets(n_shapelets_per_size=shapelet_sizes,
                        optimizer="adam",
                        weight_regularizer=.01,
                        max_iter=1000,
                        verbose=1)

In [ ]:
# Fit the shapelet model
shp_clf.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import accuracy_score
# Evaluate the shapelet classifier
predicted_labels = shp_clf.predict(X_test)
print("Correct classification rate:", accuracy_score(y_test, predicted_labels))
predicted_locations = shp_clf.locate(X)

In [ ]:
# Evaluate shapelet-based model
shp_results, (shp_y_train_pred, shp_y_test_pred,
              shp_y_train_proba, shp_y_test_proba) = evaluate_binary(
    shp_clf, X_train.reshape(X_train.shape[0], X_train.shape[1]),
    y_train,
    X_test.reshape(X_test.shape[0], X_test.shape[1]),
    y_test
)
pd.Series(shp_results).sort_index()

In [ ]:
#Confusion matrix (test) for shapelet model
cm_shp = confusion_matrix(y_test, shp_y_test_pred, labels=[0, 1])
disp_shp = ConfusionMatrixDisplay(confusion_matrix=cm_shp, display_labels=[0, 1])
disp_shp.plot()
plt.title("Shapelet Model - Confusion Matrix (test)")
plt.show()

#Print classification report (test) for shapelet model
print("\nShapelet Model Classification report (test):")
print(classification_report(y_test, shp_y_test_pred, target_names=["Non-ischemic (0)", "Ischemic (1)"]))

#ROC curve (test) for shapelet model
RocCurveDisplay.from_predictions(y_test, shp_y_test_proba)
plt.title("Shapelet Model - ROC curve (test)")
plt.show()

The model achieved an overall accuracy of $49\%$, which is slightly below the threshold of random guessing in a balanced binary classification task. This regression suggests that the extracted shapelets—designed to capture local discriminative features such as QRS complexes or T-waves—completely failed to establish a valid separation boundary between the Ischemic and Non-Ischemic classes.For the Ischemic class (1), the model's performance is critically low, correctly identifying only $34\%$ of the positive cases (Recall = $0.34$). This indicates that the classifier misses nearly two-thirds of the patients with pathology. Additionally, a Precision of $0.47$ reveals that even when the model flags a patient as Ischemic, the prediction is incorrect more than half the time ($53\%$). The resulting F1-score of $0.39$ confirms that the model is unreliable for disease detection.In contrast, for the Non-Ischemic class (0), the model displayed a distinct bias, achieving a Recall of $64\%$. This skew suggests the classifier is defaulting to the "healthy" label in uncertain regions of the decision space, maximizing specificity at the expense of sensitivity. However, even with this bias, the discriminative power remains poor, as the Precision for the Non-Ischemic class is only $0.51$, implying virtually no predictive confidence.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Visualize shapelet matches on a specific time series
ts_id = 0
n_shapelets = sum(shapelet_sizes.values())

# Set seaborn style
sns.set_theme(style="whitegrid")

fig, ax = plt.subplots(figsize=(14, 6), dpi=300)

# Plot the original time series
ax.plot(X[ts_id].ravel(), label='ECG Signal', color=sns.color_palette("deep")[0], alpha=0.7)

# Plot each shapelet at its matched location
colors = sns.color_palette("husl", n_colors=len(shp_clf.shapelets_))
for idx_shp, shp in enumerate(shp_clf.shapelets_):
    t0 = predicted_locations[ts_id, idx_shp]
    ax.plot(np.arange(t0, t0 + len(shp)), shp.ravel(), 
            linewidth=2.5, color=colors[idx_shp], 
            label=f'Shapelet {idx_shp + 1} (len={len(shp)})')

ax.set_title(f'Shapelet Matches on Time Series (ID={ts_id}) — {n_shapelets} shapelets extracted', 
             fontsize=14, fontweight='bold')
ax.set_xlabel('Time (samples)', fontsize=12)
ax.set_ylabel('Amplitude (scaled)', fontsize=12)


sns.despine()
plt.tight_layout()
plt.show()